# Objective 2 – Linear Regression
- **LR 2.1:** predicting the goal difference of each match (104 rows)
- **LR 2.2:** predicting the goals scored by a team in a match (208 rows)

## Part 1 – Data preparation

In [1]:
import random
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

PERMA_SEED = 42
random.seed(PERMA_SEED)
np.random.seed(PERMA_SEED)

# connect to the database and load the tables we need
conn = sqlite3.connect("../db/fifa2026.db")

matches = pd.read_sql("SELECT * FROM matches", conn)
teams = pd.read_sql("SELECT * FROM teams", conn)
squads = pd.read_sql("SELECT * FROM squads_and_players", conn)
venues = pd.read_sql("SELECT * FROM venues", conn)
stages = pd.read_sql("SELECT * FROM tournament_stages", conn)

print(matches.shape, teams.shape, squads.shape, venues.shape, stages.shape)

(104, 17) (48, 8) (1248, 10) (16, 8) (7, 3)


In [2]:
# age of each player on the opening day (11 June 2026)
squads["age"] = (pd.to_datetime("2026-06-11") - pd.to_datetime(squads["date_of_birth"])).dt.days / 365.25

# one row per team
team_info = squads.groupby("team_id").agg(
    squad_value=("market_value_eur", "sum"),
    avg_age=("age", "mean"),
    total_caps=("caps", "sum"),
    avg_height=("height_cm", "mean"),
).reset_index()

# international goals scored by each team's forwards
fwd_goals = squads[squads["position"] == "FWD"].groupby("team_id")["goals"].sum().reset_index(name="fwd_goals")

team_info = team_info.merge(fwd_goals, on="team_id")
team_info = team_info.merge(teams[["team_id", "team_name", "fifa_code", "elo_rating"]], on="team_id")

print(team_info.shape)
team_info.sort_values("squad_value", ascending=False).head()

(48, 9)


,team_id,squad_value,avg_age,total_caps,avg_height,fwd_goals,team_name,fifa_code,elo_rating
28,29,2134173029,26.747433,758,181.730769,78,Spain,ESP,2120
32,33,1546204469,27.001000,816,184.923077,83,France,FRA,2100
44,45,1321818896,27.273206,853,184.615385,126,England,ENG,2050
40,41,1145908765,28.035276,1176,181.500000,188,Portugal,POR,2010
36,37,1025039444,29.118201,1267,179.576923,179,Argentina,ARG,2150


In [3]:
# add stage (group / knockout) and venue details to each match
matches = matches.merge(stages[["stage_id", "is_knockout"]], on="stage_id")
matches = matches.merge(venues[["venue_id", "country", "elevation_meters"]], on="venue_id")
matches["kickoff"] = pd.to_datetime(matches["date"] + " " + matches["kickoff_time_utc"])

# each match has two teams, so make one row for each team's side of the game
home = pd.DataFrame({
    "match_id": matches["match_id"], "kickoff": matches["kickoff"],
    "team_id": matches["home_team_id"], "opp_id": matches["away_team_id"],
    "goals_for": matches["home_score"], "goals_against": matches["away_score"],
    "xg_for": matches["home_xg"], "xg_against": matches["away_xg"],
    "is_home": 1, "is_knockout": matches["is_knockout"],
    "venue_country": matches["country"], "elevation": matches["elevation_meters"],
})
away = pd.DataFrame({
    "match_id": matches["match_id"], "kickoff": matches["kickoff"],
    "team_id": matches["away_team_id"], "opp_id": matches["home_team_id"],
    "goals_for": matches["away_score"], "goals_against": matches["home_score"],
    "xg_for": matches["away_xg"], "xg_against": matches["home_xg"],
    "is_home": 0, "is_knockout": matches["is_knockout"],
    "venue_country": matches["country"], "elevation": matches["elevation_meters"],
})

team_matches = pd.concat([home, away]).sort_values(["team_id", "kickoff"]).reset_index(drop=True)
print(team_matches.shape)

(208, 12)


### Form from earlier matches only
For each match we use the team's average xG from its **previous** matches: (total xG so far − this match's xG) ÷ number of earlier matches.
A team's first match has no history, so it gets a neutral value of 1.3. Rest days for the first match are set to 7.

In [4]:
by_team = team_matches.groupby("team_id")
games_before = by_team.cumcount()

# average xG from earlier matches only (this match is subtracted out)
team_matches["prev_xg_for"] = (by_team["xg_for"].cumsum() - team_matches["xg_for"]) / games_before
team_matches["prev_xg_against"] = (by_team["xg_against"].cumsum() - team_matches["xg_against"]) / games_before
team_matches["rest_days"] = by_team["kickoff"].diff().dt.days

# first match of each team: no history yet
team_matches["prev_xg_for"] = team_matches["prev_xg_for"].fillna(1.3)
team_matches["prev_xg_against"] = team_matches["prev_xg_against"].fillna(1.3)
team_matches["rest_days"] = team_matches["rest_days"].fillna(7)

# add squad stats, and mark host nations playing in their own country
team_matches = team_matches.merge(team_info, on="team_id")
team_matches["is_host"] = (team_matches["fifa_code"] == team_matches["venue_country"]).astype(int)

team_matches[["team_name", "match_id", "xg_for", "prev_xg_for", "rest_days", "is_host"]].head(6)

,team_name,match_id,xg_for,prev_xg_for,rest_days,is_host
0,Mexico,1,1.84,1.300000,7.0,1
1,Mexico,25,1.62,1.840000,6.0,1
2,Mexico,49,1.27,1.730000,6.0,1
3,Mexico,79,1.02,1.576667,6.0,1
4,Mexico,92,1.88,1.437500,4.0,1
5,South Africa,1,0.52,1.300000,7.0,0
